# Lab 3 — Millions of TVs, Who Is Watching?
Name: Andrii Bohdan | Group: KM-43

## Setup

In [ ]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
DATA = kagglehub.dataset_download("shkarupylomaxim/adtech-lab1-tv-panel/versions/4")
programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")
acr_sessions = pd.read_parquet(f"{DATA}/acr_sessions.parquet")
hh_attributes = pd.read_parquet(f"{DATA}/hh_attributes.parquet")
reference_ratings = pd.read_parquet(f"{DATA}/reference_ratings.parquet")

## Task 1.1 — Five questions
**Answer:**

| Question | `acr_sessions` (smart-TV ACR) | Lab 1 `viewings` (people meter panel) |
|---|---|---|
| **Grain** | one row = one continuous session on one TV (one network, program or input) | one row = one viewing session of one panel member (or household) on one of the ten networks |
| **Key** | `session_id`; `tv_id` for the device; `hashed_ip` is the only link to households | `viewing_id`; `respondent_id` links to `daily_weights` and `viewer_attributes` |
| **Coverage** | 29,940 opted-in TVs of one brand, one set per home; everything the screen shows (any US network, apps, unknown), Oct 14–27 | about 4,250 recruited homes, every set and every person in them; only the ten networks; October |
| **Weight** | none: every TV counts once, so all results are unweighted | daily weights that project in-tab homes and persons to 125.8 M US TV households and 317 M persons |
| **Provenance** | byproduct of the TV: screen fingerprints matched to a library of live feeds, homes that clicked "agree" | a measurement system: probability-recruited panel, meters, persons pressing buttons, weighting to census targets |

The biggest difference is **weight** (together with how the homes got in): the panel is a designed sample weighted to represent the US, while the ACR footprint is a convenience sample of whoever bought this brand and opted in, with no weights at all. Its size cannot fix that.

## Task 1.2 — Size and shape

In [ ]:
import re
import numpy as np


def to_naive(s):
    s = pd.to_datetime(s)
    if getattr(s.dt, "tz", None) is not None:
        s = s.dt.tz_localize(None)
    return s


def to_eastern(s):
    """UTC timestamps -> naive US Eastern clock time."""
    return pd.to_datetime(s, utc=True).dt.tz_convert("US/Eastern").dt.tz_localize(None)


# panel files as in Lab 1 (naive US Eastern times)
for col in ["start_ts", "end_ts", "broadcast_date"]:
    programs[col] = to_naive(programs[col])
programs["broadcast_date"] = programs["broadcast_date"].dt.normalize()
for col in ["start_ts", "end_ts"]:
    viewings[col] = to_naive(viewings[col])
daily_weights["broadcast_date"] = to_naive(daily_weights["broadcast_date"]).dt.normalize()

# ACR sessions: UTC -> Eastern, broadcast day 06:00-05:59
DAYS = pd.date_range("2024-10-14", "2024-10-27")
acr = acr_sessions.copy()
acr["start"] = to_eastern(acr["start_utc"])
acr["end"] = to_eastern(acr["end_utc"])
acr["bdate"] = (acr["start"] - pd.Timedelta("6h")).dt.normalize()
acr = acr[acr["bdate"].isin(DAYS)].copy()
acr["hours"] = (acr["end"] - acr["start"]).dt.total_seconds() / 3600
acr["session_type"] = acr["session_type"].astype(str).str.strip().str.lower()
acr["network_code"] = acr["network_code"].fillna("").astype(str).str.strip()

# edit rule: drop sessions shorter than 1 minute
s = acr[acr["hours"] * 60 >= 1].copy()

raw_tv_days = acr.groupby("bdate")["tv_id"].nunique()
intab = s.groupby("bdate")["tv_id"].nunique()          # TVs in-tab per broadcast day

print("TVs:", acr_sessions["tv_id"].nunique())
print("Sessions:", len(acr_sessions))
print("Broadcast days:", s["bdate"].nunique())
print("Average TVs in-tab per day:", f"{intab.mean():.1f}")
print("Hours per TV per day, raw:", f"{acr['hours'].sum() / raw_tv_days.sum():.2f}")
print("Hours per TV per day, after edit rule:", f"{s['hours'].sum() / intab.sum():.2f}")
print("Share of sessions dropped:", f"{1 - len(s) / len(acr):.3f}")

## Task 1.3 — What the TV could not see

In [ ]:
by_type = (s.groupby("session_type")["hours"].sum() / s["hours"].sum()).round(3).rename("Share of hours")
by_input = (s.groupby("input_source")["hours"].sum() / s["hours"].sum()).round(3).rename("Share of hours")
display(by_type.sort_values(ascending=False).to_frame())
display(by_input.sort_values(ascending=False).to_frame())
display((s.pivot_table(index="input_source", columns="session_type", values="hours", aggfunc="sum", fill_value=0)
         / s["hours"].sum()).round(3))
print("Share of TV-on hours unknown (%):", f"{s.loc[s['session_type'] == 'unknown', 'hours'].sum() / s['hours'].sum() * 100:.1f}")

**Answer:** ACR only recognizes what is in its reference library, which is built from live linear feeds (and the TV's own apps report themselves). A real feed sees much more of the TV's time as unknown because far more viewing today goes around that library: streaming through an external device (Roku, Fire TV stick, Apple TV) plugged into HDMI, where the TV sees a picture but no matching live feed and no app of its own; video games and Blu-ray/DVD; on-demand and older DVR content, local or regional channels, FAST channels and ads that are not in the library; menus, screensavers and overlays that break the fingerprint. In this synthetic feed the TV's built-in apps and most cable/antenna viewing are recognized, so its unknown share is lower than LG's 66%. Two kinds of viewing that end up as unknown here: streaming apps on an external streaming device (input `streaming_device`) and gaming or disc playback (`game_or_disc`).

## Task 2.1 — Match

In [ ]:
# one IP per TV (the most frequent one, if a TV shows several)
tv_ip = (acr_sessions.groupby(["tv_id", "hashed_ip"]).size().reset_index(name="n")
         .sort_values("n").drop_duplicates("tv_id", keep="last").set_index("tv_id")["hashed_ip"])
tv_ip = tv_ip.reindex(acr_sessions["tv_id"].unique())
hh_per_ip = hh_attributes.groupby("hashed_ip")["household_id"].nunique()
n_hh = tv_ip.map(hh_per_ip).fillna(0)
match_status = pd.Series(np.select([n_hh == 1, n_hh > 1], ["matched", "shared"], "none"), index=tv_ip.index)
shares = match_status.value_counts(normalize=True) * 100

print("Matched to one household (%):", f"{shares.get('matched', 0):.1f}")
print("Shared IP, unresolved (%):", f"{shares.get('shared', 0):.1f}")
print("Not matched (%):", f"{shares.get('none', 0):.1f}")

# matched TVs with their household
one_ip = hh_attributes[hh_attributes["hashed_ip"].map(hh_per_ip) == 1]
matched = (tv_ip[match_status == "matched"].rename("hashed_ip").reset_index()
           .rename(columns={"index": "tv_id"}).merge(one_ip, on="hashed_ip", how="left"))

**Answer:** An IP address identifies a router connection, not a household. Many homes behind one public IP (apartment buildings, dorms, carrier-grade NAT of mobile and cable providers) cannot be told apart, which is the "shared" group. And one home's IP changes over time: providers reassign dynamic IPs, people move, use a VPN or a hotspot, so the IP the TV reports today may not be the one the data vendor saw when it compiled the file, which ends up in "not matched". On top of that the IP is hashed, so no fuzzy matching is possible: either the hashes are identical or there is no link. Every unmatched or unresolved TV is a home whose demographics we simply do not know.

## Task 2.2 — Profile

In [ ]:
# panel households in-tab Oct 14-27, each with its average weight over those days
hdw = daily_weights[(daily_weights["resp_type"] == "H") & daily_weights["broadcast_date"].isin(DAYS)]
hh_w = hdw.groupby("respondent_id")["weight"].mean()

va = viewer_attributes.merge(attributes, on="attr_id", how="left")
hva = va[va["respondent_id"].isin(hh_w.index)]


def panel_attr(keyword):
    """Label of one household attribute group (e.g. region) for every panel household."""
    r = hva[hva["attr_group"].astype(str).str.contains(keyword, case=False, na=False)]
    label = r["attr_name"] if r["attr_name"].nunique() > 1 else r["value"]
    return label.astype(str).str.strip().groupby(r["respondent_id"]).first()


def income3(label):
    """Six income bands -> three groups."""
    t = str(label).lower().replace(",", "")
    nums = [float(x) for x in re.findall(r"\d+\.?\d*", t)]
    if not nums:
        return np.nan
    x = nums[0] / 1000 if nums[0] >= 1000 else nums[0]
    if "under" in t or "<" in t or "less" in t:
        return "Under $50k" if x <= 50 else np.nan
    return "Under $50k" if x < 50 else ("$50-100k" if x < 100 else "$100k+")


panel_region = panel_attr("region")
panel_income = panel_attr("income").map(income3)
print("Panel region labels:", sorted(panel_region.unique()))
print("Panel income labels:", sorted(panel_attr("income").unique()))

matched["income3"] = matched["income_band"].map(income3)
rows = []
for kind, fp, pn, order in [
    ("Region", matched["region"], panel_region, ["Northeast", "Midwest", "South", "West"]),
    ("Income", matched["income3"], panel_income, ["Under $50k", "$50-100k", "$100k+"]),
]:
    fp_share = fp.value_counts(normalize=True)
    w = hh_w.reindex(pn.index)
    pn_share = w.groupby(pn).sum() / w.sum()
    for g in order:
        rows.append((kind, g, fp_share.get(g, 0), pn_share.get(g, 0)))
profile = pd.DataFrame(rows, columns=["Kind", "Group", "Footprint share", "Panel share"])
profile["Index"] = (profile["Footprint share"] / profile["Panel share"] * 100).round(0)
profile[["Footprint share", "Panel share"]] = profile[["Footprint share", "Panel share"]].round(3)
display(profile)
print("Over-represented (index > 110):", list(profile.loc[profile["Index"] > 110, "Group"]))
print("Under-represented (index < 90):", list(profile.loc[profile["Index"] < 90, "Group"]))

# head of household age: matched TVs vs all households in the compiled file (a check for Task 2.3)
age_check = pd.DataFrame({
    "Matched TVs": matched["head_age_band"].value_counts(normalize=True),
    "All households in file": hh_attributes["head_age_band"].value_counts(normalize=True),
}).sort_index()
age_check["Index"] = (age_check["Matched TVs"] / age_check["All households in file"] * 100).round(0)
display(age_check.round(3))

**Answer:** The footprint is not a miniature US. Higher-income homes ($100k+) are over-represented and homes under $50k are under-represented (index well below 100); the regions printed with an index above 110 are over-represented, those below 90 under-represented. The head-of-household check points the same way: young and middle-aged heads are over-represented and 65+ homes are under-represented. A reason: to be in the footprint a home must own a recent smart TV of this brand, have it connected to broadband, and leave ACR switched on (or not opt out). New, connected TVs are bought more often by richer, younger, tech-savvy households in larger metro areas; older and low-income homes keep older sets, use them without internet, or turn tracking off.

## Task 2.3 — Ratings

In [ ]:
live = s[s["session_type"] == "live"]


def tv_watchers(t):
    """TVs whose live sessions on the telecast's network overlap it by at least 1 minute (in-tab that day)."""
    v = live[(live["network_code"] == t.network_code) & (live["end"] > t.start_ts) & (live["start"] < t.end_ts)]
    st = v["start"].where(v["start"] > t.start_ts, t.start_ts)
    en = v["end"].where(v["end"] < t.end_ts, t.end_ts)
    ov = ((en - st).dt.total_seconds() / 60).groupby(v["tv_id"]).sum()
    ids = set(ov[ov >= 1].index)
    return ids & set(s.loc[s["bdate"] == t.broadcast_date, "tv_id"])


ref = reference_ratings.merge(
    programs[["telecast_id", "network_code", "program_name", "broadcast_date", "start_ts", "end_ts"]],
    on="telecast_id", how="left")
print("Ten-network codes found in ACR:", sorted(set(programs["network_code"]) & set(s["network_code"])))
ref["TVs watched"] = [len(tv_watchers(t)) for t in ref.itertuples()]
ref["TVs in-tab"] = ref["broadcast_date"].map(intab).values
ref["TVs' HH rating"] = ref["TVs watched"] / ref["TVs in-tab"] * 100
ref["Ratio"] = ref["TVs' HH rating"] / ref["hh_rating"]
ratings = ref[["telecast_id", "network_code", "program_name", "broadcast_date", "TVs' HH rating", "hh_rating", "Ratio"]].rename(
    columns={"hh_rating": "Reference HH rating"})
ratings["broadcast_date"] = ratings["broadcast_date"].dt.strftime("%Y-%m-%d")
ratings[["TVs' HH rating", "Reference HH rating"]] = ratings[["TVs' HH rating", "Reference HH rating"]].round(1)
ratings["Ratio"] = ratings["Ratio"].round(3)
display(ratings.sort_values("Ratio"))
print("Average ratio:", f"{ref['Ratio'].mean():.3f}")
print("Furthest off:", ref.loc[ref["Ratio"].idxmin(), "program_name"])
print("Closest:", ref.loc[ref["Ratio"].idxmax(), "program_name"])

**Answer:** The TVs' ratings are too low for every telecast (all ratios are below 1). Part of that hits every program the same way: the reference counts a home that watched on any set, while the footprint sees only one TV per home, and the in-tab denominator includes homes that were on apps or unknown sources (Task 3 looks at this). The differences between telecasts are the interesting part: the furthest off are the programs with older audiences (Hannity on Fox News, NCIS on CBS), the closest are the younger ones (Grotesquerie on FX, Real Housewives on Bravo), with sports and the lifestyle networks in between. That fits the profile: the footprint has too few older (65+) and lower-income homes, which are exactly the homes that watch Fox News and CBS, so those programs lose the most audience; programs that young, connected homes watch are measured almost correctly. Without weights, the bias follows the demography of each program's audience.

## Task 3.1 — Hours per home

In [ ]:
TEN = sorted(programs["network_code"].unique())

# TVs: live sessions on the ten networks
tv_live10 = live[live["network_code"].isin(TEN)]
tv_hours = tv_live10["hours"].sum() / intab.sum()

# panel: H rows on the ten networks, live viewing, in-tab households, daily weights
hv = viewings[viewings["resp_type"] == "H"].copy()
if "playback" in hv.columns:
    is_live = hv["playback"].astype(str).str.strip().str.lower() == "live"
    if is_live.any():
        hv = hv[is_live]
hv["broadcast_date"] = (hv["start_ts"] - pd.Timedelta("6h")).dt.normalize()
hv["hours"] = (hv["end_ts"] - hv["start_ts"]).dt.total_seconds() / 3600
hv = hv[hv["network_code"].isin(TEN)].drop(columns="weight", errors="ignore").merge(
    hdw[["respondent_id", "broadcast_date", "weight"]], on=["respondent_id", "broadcast_date"], how="inner")
panel_hours = (hv["hours"] * hv["weight"]).sum() / hdw["weight"].sum()

print("Live hours per TV per day (ten networks):", f"{tv_hours:.2f}")
print("Live hours per panel household per day (ten networks):", f"{panel_hours:.2f}")
print("TVs / panel:", f"{tv_hours / panel_hours:.3f}")

**Answer:** The smart TVs show clearly fewer hours than the panel homes (see the ratio above). The panel meters every set in the home and counts a household as viewing when any of them is on, while the ACR footprint sees one set: an average US home has about 2.6 TVs, only about 1.1 of them of the same brand, so the viewing on the bedroom or kitchen TV, or the older non-smart set, is invisible. Part of the footprint TV's own time is also not counted here: it goes to apps, other networks outside the ten, and unknown sources. Smaller effects: the footprint homes are younger and richer, who watch less linear TV, and the edit rule removes short tuning that the panel may still credit.

## Task 3.2 — Network shares

In [ ]:
net_share = pd.DataFrame({
    "Share in TVs": tv_live10.groupby("network_code")["hours"].sum() / tv_live10["hours"].sum(),
    "Share in panel": (hv["hours"] * hv["weight"]).groupby(hv["network_code"]).sum() / (hv["hours"] * hv["weight"]).sum(),
}).reindex(TEN).fillna(0)
net_share["Index"] = (net_share["Share in TVs"] / net_share["Share in panel"] * 100).round(0)
net_share = net_share.round(3).sort_values("Index")
display(net_share)
print("Over-stated by the TVs (index > 110):", list(net_share.index[net_share["Index"] > 110]))
print("Under-stated by the TVs (index < 90):", list(net_share.index[net_share["Index"] < 90]))

## Task 3.3 — What each can see
**Answer:**

What the **panel** can see and this data cannot:
1. **People**: who is in the room (age, sex, guests), so person ratings and demographic targets like A25–54; ACR knows only that a TV was on a channel.
2. **Every set in the home**, of every brand, smart or not, connected or not, so a true household rating and all viewing of the home.
3. **A representative, weighted sample** that includes the homes ACR misses: older and low-income homes, homes without a smart TV or broadband, homes that opted out.

What this **ACR data** can see and the panel's ten-network files cannot:
1. **Everything on the screen**: all US networks, not just ten, plus apps, streaming devices, games and the share of time the TV is on but unrecognized.
2. **Scale**: about 30,000 TVs, so small cable networks, single episodes, local markets (DMA) and individual ad spots get enough homes to measure, where the panel has zero-cells.
3. **Second-by-second detail and the input source**: exact switching moments, ad-break drop-off and whether the picture came from a cable box, antenna or streaming device.

## Task 4.1 — Daily plots

In [ ]:
BIG = ["CBS", "NBC", "ABC", "FOX", "FNC"]
big_live = live[live["network_code"].isin(BIG)]
daily_net = big_live.pivot_table(index="bdate", columns="network_code", values="hours", aggfunc="sum").reindex(DAYS).fillna(0)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(intab.index, intab.values, marker="o", color="#1f5aa6")
axes[0].set_ylabel("TVs in-tab")
axes[0].set_title("TVs in-tab per broadcast day")
axes[0].grid(alpha=0.3)
for net in BIG:
    if net in daily_net:
        axes[1].plot(daily_net.index, daily_net[net].values / 1000, marker="o", label=net)
axes[1].set_ylabel("Identified live hours, thousands")
axes[1].set_title("Identified live hours per broadcast day, five big networks")
axes[1].grid(alpha=0.3)
axes[1].legend()
axes[1].set_xticks(DAYS)
axes[1].set_xticklabels([d.strftime("%a %m-%d") for d in DAYS], rotation=45)
plt.tight_layout()
plt.show()

## Task 4.2 — Spike and broken day

In [ ]:
# spike: the network-day with the most identified live hours, and what aired
spike_day, spike_net = daily_net.stack().idxmax()
tel = programs[(programs["network_code"] == spike_net) & (programs["broadcast_date"] == spike_day)].copy()
tel["TVs watched"] = [len(tv_watchers(t)) for t in tel.itertuples()]
display(tel.sort_values("TVs watched", ascending=False)[["program_name", "start_ts", "end_ts", "TVs watched"]].head(5))

# broken day: compare every day with the median day
daily = pd.DataFrame({
    "TVs in-tab": intab,
    "Hours per in-tab TV": s.groupby("bdate")["hours"].sum() / intab,
    "Identified live share": live.groupby("bdate")["hours"].sum() / s.groupby("bdate")["hours"].sum(),
    "Unknown share": s[s["session_type"] == "unknown"].groupby("bdate")["hours"].sum() / s.groupby("bdate")["hours"].sum(),
}).reindex(DAYS)
dev = (daily / daily.median() - 1)
# a real event (the spike) raises viewing; a data problem moves in-tab TVs or hours per in-tab TV
broken = dev.drop(index=spike_day)[["TVs in-tab", "Hours per in-tab TV", "Unknown share"]].abs().max(axis=1).idxmax()
display(daily.assign(**{"Biggest deviation from median": dev.abs().max(axis=1)}).round(3))

print("Biggest spike (date, network):", spike_day.strftime("%Y-%m-%d"), spike_net)
print("Top program that day:", tel.loc[tel["TVs watched"].idxmax(), "program_name"])
print("Broken day (date):", broken.strftime("%Y-%m-%d"))
print("What moved on the broken day (vs median day):", dev.loc[broken].round(3).to_dict())

**Answer:** The biggest spike is live sport, the program printed above: the World Series on FOX (from Friday, October 25) or the Sunday NFL games, which keep millions of homes on one network for three hours or more, so identified hours on that network jump far above a normal day.

The broken day is the date printed above. Its change is in the data, not in the audience: the row of the table shows which measure moved while viewing on the days around it is normal. If the number of TVs in-tab collapses, part of the feed did not arrive (a missing file or an outage of the collection): every daily total (hours, impressions) drops although nobody stopped watching, and a rating is wrong if the missing TVs are not random. If TVs stay in-tab but their hours per TV or the identified share collapse (the unknown share jumps), the recognition failed: the TVs still count in the denominator but their viewing is not credited to any network, so every rating that day comes out too low. A planner who does not notice would read it as a real ratings drop or a lost audience. I would **flag** the day and keep it out of averages, trends and ratings (dropping it from the reported series) rather than "fix" it, because we cannot know the true viewing of the missing TVs or sessions; the raw rows stay in the data with the flag so the vendor can redeliver them.

## Task 5.1 — Sampling error vs bias

In [ ]:
h_intab = hdw.groupby("broadcast_date")["respondent_id"].nunique()
pick = ref[ref["program_name"].str.contains("Football|Hannity|Unsellable", case=False, na=False)].copy()
p = pick["hh_rating"] / 100
pick["n TVs"] = pick["TVs in-tab"]
pick["n panel HH"] = pick["broadcast_date"].map(h_intab).values
pick["SE footprint"] = np.sqrt(p * (1 - p) / pick["n TVs"]) * 100
pick["SE panel"] = np.sqrt(p * (1 - p) / pick["n panel HH"]) * 100
pick["SE panel / SE footprint"] = pick["SE panel"] / pick["SE footprint"]
pick["TVs' rating - reference"] = pick["TVs' HH rating"] - pick["hh_rating"]
se_table = pick[["program_name", "hh_rating", "n TVs", "n panel HH", "SE footprint", "SE panel",
                 "SE panel / SE footprint", "TVs' rating - reference"]].rename(columns={"hh_rating": "Reference"})
display(se_table.round(3))

**Answer:** With about 30,000 TVs instead of about 4,000 panel homes, the footprint's sampling error is roughly the square root of the size ratio smaller (see the last-but-one column, about 2.5–3 times), only a few hundredths of a rating point. But the gap between the TVs' rating and the reference is many times larger than either standard error: it is bias, not noise. Bias does not shrink with more data: 300,000 TVs of the same kind of homes would give an even smaller SE around the same wrong number. A big sample is precise, not accurate; only representativeness (or weighting, Lab 4) removes the bias.

## Task 5.2 — Answer to the planner
**Answer:** From this feed alone I would not give a Sunday Night Football rating for the client deck: its unweighted household rating for Dallas at San Francisco is clearly below the currency number (Task 2.3), so it would understate the audience the client actually bought. The number I can give today is the currency (reference) household rating; the feed can only be shown as a supporting, directional figure, clearly labeled "unweighted smart-TV footprint, not currency". The feed sees one TV per home and no people, so it cannot tell who watched or count viewing on the other sets in the house; it also misses viewing through apps, streaming sticks and unrecognized sources, and all homes that do not own this brand, opted out, or could not be matched to a household (unmatched and shared-IP TVs). Its homes are richer and younger than the US, so the error is different for every program and a single correction factor would not work. Before using it for planning I would need weights (calibrating the footprint to the panel or census by age, income and region — Lab 4), a validation of the weighted ratings against the currency over many telecasts, a persons model to get from households to the client's demo, and a check for broken delivery days like the one found in Task 4.

## Bonus

In [ ]:
def panel_viewers(t, rtypes):
    """Panel respondents who watched telecast t for at least 1 minute (Lab 1 rule)."""
    v = viewings[viewings["resp_type"].isin(rtypes) & (viewings["network_code"] == t.network_code)
                 & (viewings["end_ts"] > t.start_ts) & (viewings["start_ts"] < t.end_ts)]
    st = v["start_ts"].where(v["start_ts"] > t.start_ts, t.start_ts)
    en = v["end_ts"].where(v["end_ts"] < t.end_ts, t.end_ts)
    ov = ((en - st).dt.total_seconds() / 60).groupby(v["respondent_id"]).sum()
    return set(ov[ov >= 1].index)


snf = ref[ref["program_name"].str.contains("Football", case=False, na=False)
          & (ref["network_code"] == "NBC")].iloc[0]
day_w = daily_weights[daily_weights["broadcast_date"] == snf["broadcast_date"]].set_index("respondent_id")["weight"]
day_types = daily_weights[daily_weights["broadcast_date"] == snf["broadcast_date"]].set_index("respondent_id")["resp_type"]

person_age = (va[va["attr_id"] == 300].drop_duplicates("respondent_id").set_index("respondent_id")["value"].astype(float))
a2554 = set(person_age[person_age.between(25, 54)].index)
persons_today = day_w[day_types.isin(["R", "L"])]
a2554_today = persons_today[persons_today.index.isin(a2554)]
hh_today = day_w[day_types == "H"]

hh_watched = hh_today.reindex(list(panel_viewers(snf, ["H"]))).dropna()
a_watched = a2554_today.reindex(list(panel_viewers(snf, ["R", "L"]))).dropna()
vpvh = a_watched.sum() / hh_watched.sum()
a2554_rating = snf["TVs' HH rating"] * vpvh * (hh_today.sum() / a2554_today.sum())

print("VPVH A25-54:", f"{vpvh:.3f}")
print("A25-54 rating (persons):", f"{a2554_rating:.1f}")
print("Panel A25-54 rating, for comparison:", f"{a_watched.sum() / a2554_today.sum() * 100:.1f}")

**Answer:** The estimate takes the household rating from the TVs and puts the panel's people into those homes. It assumes that a footprint home that watched Sunday Night Football has, on average, the same number of adults 25–54 in front of the TV as a panel viewing home, i.e. that footprint homes are like US homes in household size and age mix, and that one TV per home captures the household's viewing. Both are false here: the footprint's homes are younger and richer than the US, and the TV-based household rating is already biased low, so the A25–54 number inherits that bias (it is multiplied, not corrected) and is only as good as the household rating underneath it.